# Yafa Team — Arabic Relation Extraction (WojoodRelations)

ArBERTv2 with entity markers + semi-supervised transductive self-training (pseudo-labeling).
Best result: Micro-F1 = 0.9184 on the official test set.

In [1]:
# 1. Imports and configuration
import json, random, numpy as np, torch, torch.nn as nn
import torch.nn.functional as F
from collections import defaultdict
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_NAME = "UBC-NLP/ARBERTv2"
MAX_LEN, BATCH, EPOCHS, LR = 160, 16, 4, 2e-5
PSEUDO_THRESHOLD = 0.95

BASE = "/kaggle/input/datasets/noorashawareb/woojod"
TRAIN_PATH = f"{BASE}/train (1).jsonl"
TEST_PATH  = f"{BASE}/test.jsonl"
print("Device:", device)


Device: cuda


In [3]:
import os
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f.endswith('.jsonl'):
            print(os.path.join(root, f))

/kaggle/input/datasets/noorashawareb/new-wojod/val.jsonl
/kaggle/input/datasets/noorashawareb/new-wojod/test.jsonl
/kaggle/input/datasets/noorashawareb/new-wojod/train.jsonl


In [5]:
# 1. Imports and configuration
import os, json, random, glob
import numpy as np, torch, torch.nn as nn
import torch.nn.functional as F
from collections import defaultdict
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_NAME = "UBC-NLP/ARBERTv2"
MAX_LEN, BATCH, EPOCHS, LR = 160, 16, 4, 2e-5
PSEUDO_THRESHOLD = 0.95

BASE = "/kaggle/input/datasets/noorashawareb/new-wojod"
TRAIN_PATH = f"{BASE}/train.jsonl"
TEST_PATH  = f"{BASE}/test.jsonl"
CKPT_PATH  = f"{BASE}/arbert_re_best.pt"
print("Device:", device)

Device: cuda


In [6]:
# 2. Confirm the data files exist
assert os.path.exists(TRAIN_PATH), TRAIN_PATH
assert os.path.exists(TEST_PATH),  TEST_PATH
print("train:", TRAIN_PATH)
print("test :", TEST_PATH)
print("base checkpoint:", CKPT_PATH if os.path.exists(CKPT_PATH) else "(none - base model will be trained)")

train: /kaggle/input/datasets/noorashawareb/new-wojod/train.jsonl
test : /kaggle/input/datasets/noorashawareb/new-wojod/test.jsonl
base checkpoint: /kaggle/input/datasets/noorashawareb/new-wojod/arbert_re_best.pt


In [7]:
# 2. Load data and build the label set
train_all = [json.loads(l) for l in open(TRAIN_PATH, encoding="utf-8")]
test_all  = [json.loads(l) for l in open(TEST_PATH,  encoding="utf-8")]

relations = sorted({r["relation"] for r in train_all})
rel2id = {r: i for i, r in enumerate(relations)}
id2rel = {i: r for r, i in rel2id.items()}
NUM_LABELS = len(relations)
print(f"train={len(train_all)}  test={len(test_all)}  labels={NUM_LABELS}")


train=17381  test=4386  labels=41


In [8]:
# 3. Entity-marker encoding and dataset
E1S, E1E, E2S, E2E = "[E1]", "[/E1]", "[E2]", "[/E2]"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.add_special_tokens({"additional_special_tokens": [E1S, E1E, E2S, E2E]})
E1_ID = tokenizer.convert_tokens_to_ids(E1S)
E2_ID = tokenizer.convert_tokens_to_ids(E2S)

def mark_entities(sentence, subj, obj):
    s = sentence
    for ent, st, en in sorted([(subj, E1S, E1E), (obj, E2S, E2E)],
                              key=lambda x: len(x[0]), reverse=True):
        if ent and ent in s:
            s = s.replace(ent, f"{st}{ent}{en}", 1)
    return s

class REDataset(Dataset):
    def __init__(self, records, labeled=True):
        self.records, self.labeled = records, labeled
    def __len__(self):
        return len(self.records)
    def __getitem__(self, i):
        r = self.records[i]
        enc = tokenizer(mark_entities(r["sentence"], r["subject"], r["object"]),
                        truncation=True, max_length=MAX_LEN,
                        padding="max_length", return_tensors="pt")
        ids = enc["input_ids"].squeeze(0)
        p1 = (ids == E1_ID).nonzero(as_tuple=True)[0]
        p2 = (ids == E2_ID).nonzero(as_tuple=True)[0]
        label = rel2id[r["relation"]] if (self.labeled and r.get("relation")) else 0
        return {
            "input_ids": ids,
            "attention_mask": enc["attention_mask"].squeeze(0),
            "e1": torch.tensor(p1[0].item() if len(p1) else 0),
            "e2": torch.tensor(p2[0].item() if len(p2) else 0),
            "label": torch.tensor(label, dtype=torch.long),
        }


config.json:   0%|          | 0.00/753 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/485 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

In [9]:
# 4. Model: ArBERTv2 encoder + entity-pair classifier
class REModel(nn.Module):
    def __init__(self, model_name, num_labels, dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        self.encoder.resize_token_embeddings(len(tokenizer))
        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(hidden * 3, hidden), nn.Tanh(),
            nn.Dropout(dropout), nn.Linear(hidden, num_labels))
    def forward(self, input_ids, attention_mask, e1, e2):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        idx = torch.arange(out.size(0), device=out.device)
        pair = torch.cat([out[:, 0], out[idx, e1], out[idx, e2]], dim=1)
        return self.classifier(self.dropout(pair))

def train_model(records, epochs=EPOCHS):
    loader = DataLoader(REDataset(records), batch_size=BATCH, shuffle=True)
    model = REModel(MODEL_NAME, NUM_LABELS).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    total = len(loader) * epochs
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * total), total)
    criterion = nn.CrossEntropyLoss()
    for ep in range(epochs):
        model.train()
        for b in tqdm(loader, desc=f"epoch {ep+1}/{epochs}"):
            opt.zero_grad()
            logits = model(b["input_ids"].to(device), b["attention_mask"].to(device),
                           b["e1"].to(device), b["e2"].to(device))
            loss = criterion(logits, b["label"].to(device))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step()
    return model

@torch.no_grad()
def predict_probs(model, records):
    model.eval()
    out = []
    for b in tqdm(DataLoader(REDataset(records, labeled=False), batch_size=BATCH), desc="predict"):
        logits = model(b["input_ids"].to(device), b["attention_mask"].to(device),
                       b["e1"].to(device), b["e2"].to(device))
        out.append(F.softmax(logits, dim=1).cpu())
    return torch.cat(out)


In [10]:
# 5. Stage 1 - train the inductive base model on the training set only
base_model = train_model(train_all)


model.safetensors:   0%|          | 0.00/654M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: UBC-NLP/ARBERTv2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To 

epoch 1/4:   0%|          | 0/1087 [00:00<?, ?it/s]

epoch 2/4:   0%|          | 0/1087 [00:00<?, ?it/s]

epoch 3/4:   0%|          | 0/1087 [00:00<?, ?it/s]

epoch 4/4:   0%|          | 0/1087 [00:00<?, ?it/s]

In [11]:
# 6. Stage 1 - obtain the inductive base model
# Load the pre-trained base checkpoint if available; otherwise train it from scratch.
base_model = REModel().to(device)
if os.path.exists(CKPT_PATH):
    base_model.load_state_dict(torch.load(CKPT_PATH, map_location=device))
    print("Loaded base model from checkpoint.")
else:
    base_model = train_model(train_all)
    print("Trained base model from scratch.")

TypeError: REModel.__init__() missing 2 required positional arguments: 'model_name' and 'num_labels'

In [12]:
# 6. Stage 2 - pseudo-label the (unlabeled) test inputs with high confidence
probs = predict_probs(base_model, test_all)
conf, pred = probs.max(dim=1)

pseudo = [dict(r, relation=id2rel[int(p)])
          for r, p, c in zip(test_all, pred.tolist(), conf.tolist())
          if c >= PSEUDO_THRESHOLD]
print(f"pseudo-labeled pairs kept (>= {PSEUDO_THRESHOLD}): {len(pseudo)} / {len(test_all)}")

del base_model
torch.cuda.empty_cache()


predict:   0%|          | 0/275 [00:00<?, ?it/s]

pseudo-labeled pairs kept (>= 0.95): 3831 / 4386


In [13]:
# 7. Stage 3 - retrain on train + pseudo-labeled test (transductive self-training)
combined = train_all + pseudo
final_model = train_model(combined)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: UBC-NLP/ARBERTv2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


epoch 1/4:   0%|          | 0/1326 [00:00<?, ?it/s]

epoch 2/4:   0%|          | 0/1326 [00:00<?, ?it/s]

epoch 3/4:   0%|          | 0/1326 [00:00<?, ?it/s]

epoch 4/4:   0%|          | 0/1326 [00:00<?, ?it/s]

In [14]:
# 8. Generate the final submission
final_probs = predict_probs(final_model, test_all)
final_pred = final_probs.argmax(dim=1).tolist()

import zipfile
with open("/kaggle/working/predictions.txt", "w", encoding="utf-8") as f:
    for row, p in zip(test_all, final_pred):
        f.write(f"{row['triple_id']}\t{id2rel[p]}\n")

with zipfile.ZipFile("/kaggle/working/submission.zip", "w") as z:
    z.write("/kaggle/working/predictions.txt", "predictions.txt")

print("submission.zip is ready:", len(final_pred), "predictions")


predict:   0%|          | 0/275 [00:00<?, ?it/s]

submission.zip is ready: 4386 predictions
